# Daily LSTM Return-Regression Overfitting Capacity by Training Window

This notebook investigates whether the daily return-regression LSTM can overfit its own training data as the training sample becomes larger. This directly follows the earlier debugging plot "Can the LSTM overfit training samples? (AAPL)", where the model was trained and evaluated on the same small training sample.

The diagnostic is intentionally in-sample. It is not designed to evaluate out-of-sample forecasting performance. Instead, it checks whether prediction collapse toward zero already happens on the training data when the sample becomes larger.

In [ ]:

import os
import shutil
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

try:
    import tensorflow as tf
    from lstm_architecture import LSTMConfig, build_lstm_model
except Exception as exc:
    raise ImportError(
        "This notebook requires the project TensorFlow/Keras environment. "
        "Select the honour-project .venv kernel before running."
    ) from exc

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "preprocessed_stock_data.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "daily_return_regression_window_sensitivity"
PLOTS_DIR = OUTPUT_DIR / "plots"

TARGET_COL = "Next_Day_Log_Return"
FEATURE_COLS = [
    "Log_Return",
    "High_Low_Range",
    "Open_Close_Log_Return",
    "Volume_Change",
    "Rolling_Vol_5",
    "Rolling_Vol_10",
    "Rolling_Vol_20",
    "Rolling_Vol_60",
    "Momentum_5",
    "Momentum_10",
    "Momentum_20",
    "Momentum_60",
    "MA_Gap_5",
    "MA_Gap_10",
    "MA_Gap_20",
    "MA_Gap_60",
    "Drawdown_20",
]

TICKERS = ["AAPL", "IBM", "MSFT"]
LOOKBACK = 60
SEED = 5
EPOCHS = 200
BATCH_SIZE = 32
TRAINING_WINDOWS = [
    ("0.5_years", 0.5),
    ("1_year", 1.0),
    ("2_years", 2.0),
    ("3_years", 3.0),
    ("5_years", 5.0),
    ("8_years", 8.0),
    ("10_years", 10.0),
    ("full_available_training_sample", None),
]

FIXED_CONFIG = LSTMConfig(
    lstm_units=(64, 32),
    dense_units=(32, 16),
    dropout=0.0,
    recurrent_dropout=0.0,
    learning_rate=1e-3,
    loss="mse",
    output_units=1,
)

tf.keras.utils.set_random_seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)
print("Model config:", FIXED_CONFIG)

Project root: d:\BA_year_3\honour project
Output directory: d:\BA_year_3\honour project\outputs\daily_return_regression_window_sensitivity
Model config: LSTMConfig(lstm_units=(64, 32), dense_units=(32, 16), dense_activation='tanh', dropout=0.0, recurrent_dropout=0.0, learning_rate=0.001, loss='mse', output_units=1)


## Load Daily Training Data

Only rows with `Split == "train"` are used. Validation and test rows are intentionally excluded from this diagnostic.

In [16]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing daily data: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, parse_dates=["Date", "Target_Date"])
required_cols = ["Ticker", "Date", "Target_Date", "Split", TARGET_COL] + FEATURE_COLS
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Daily modelling data is missing required columns: {missing_cols}")

train_df = (
    df.loc[df["Split"].eq("train"), required_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=[TARGET_COL, "Target_Date"] + FEATURE_COLS)
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

if train_df.empty:
    raise ValueError("No training rows are available after filtering.")

display(train_df.groupby("Ticker").agg(start=("Date", "min"), end=("Date", "max"), rows=("Date", "size")))

,start,end,rows
Ticker,,,
AAPL,2010-03-31,2020-12-30,2708
IBM,2010-03-31,2020-12-30,2708
MSFT,2010-03-31,2020-12-30,2708


## Clean Output Folder

The output folder is reused, but old validation/test files and plots are removed before the new in-sample diagnostic outputs are written.

In [17]:
def clean_output_folder(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    for child in output_dir.iterdir():
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()


clean_output_folder(OUTPUT_DIR)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Cleaned and recreated: {OUTPUT_DIR}")

Cleaned and recreated: d:\BA_year_3\honour project\outputs\daily_return_regression_window_sensitivity


## Helper Functions

In [18]:
def training_window_start(train_end: pd.Timestamp, years: float | None) -> pd.Timestamp:
    if years is None:
        return pd.Timestamp.min
    return train_end - pd.Timedelta(days=int(round(365.25 * years)))


def select_training_window(ticker_train_df: pd.DataFrame, years: float | None) -> pd.DataFrame:
    ticker_train_df = ticker_train_df.sort_values("Date").copy()
    train_end = ticker_train_df["Date"].max()
    if years is None:
        selected = ticker_train_df.copy()
    else:
        selected = ticker_train_df.loc[ticker_train_df["Date"].ge(training_window_start(train_end, years))].copy()
    return selected.sort_values("Date").reset_index(drop=True)


def make_training_sequences(
    ticker_df: pd.DataFrame,
    feature_columns: list[str],
    target_column: str,
    lookback: int,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    X_train, y_train, meta_train = [], [], []
    ticker_df = ticker_df.sort_values("Date").reset_index(drop=True)

    values = ticker_df[feature_columns].to_numpy(dtype=np.float32)
    targets = ticker_df[target_column].to_numpy(dtype=np.float32)

    for i in range(lookback - 1, len(ticker_df)):
        X_train.append(values[i - lookback + 1 : i + 1])
        y_train.append(targets[i])
        meta_train.append(
            {
                "Ticker": ticker_df.loc[i, "Ticker"],
                "Input_End_Date": ticker_df.loc[i, "Date"],
                "Target_Date": ticker_df.loc[i, "Target_Date"],
            }
        )

    return (
        np.asarray(X_train, dtype=np.float32),
        np.asarray(y_train, dtype=np.float32),
        pd.DataFrame(meta_train),
    )


def compute_train_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    train_actual_std = float(np.std(y_true, ddof=0))
    train_prediction_std = float(np.std(y_pred, ddof=0))
    train_variance_ratio = train_prediction_std / train_actual_std if train_actual_std > 0 else np.nan
    train_mean_abs_prediction = float(np.mean(np.abs(y_pred)))

    if train_actual_std > 0 and train_prediction_std > 0:
        corr = float(np.corrcoef(y_true, y_pred)[0, 1])
    else:
        corr = np.nan

    return {
        "train_rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "train_mae": float(mean_absolute_error(y_true, y_pred)),
        "train_r2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        "train_prediction_mean": float(np.mean(y_pred)),
        "train_prediction_std": train_prediction_std,
        "train_actual_std": train_actual_std,
        "train_variance_ratio": float(train_variance_ratio) if pd.notna(train_variance_ratio) else np.nan,
        "train_prediction_min": float(np.min(y_pred)),
        "train_prediction_max": float(np.max(y_pred)),
        "train_mean_abs_prediction": train_mean_abs_prediction,
        "train_share_abs_prediction_below_0_0005": float(np.mean(np.abs(y_pred) < 0.0005)),
        "train_direction_accuracy": float(np.mean((y_true > 0) == (y_pred > 0))),
        "train_correlation_actual_predicted": corr,
        "train_return_collapse_warning": bool(pd.notna(train_variance_ratio) and train_variance_ratio < 0.10),
        "train_near_zero_prediction_warning": bool(train_mean_abs_prediction < 0.0005),
    }


def save_plot(filename: str) -> Path:
    path = PLOTS_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()
    return path


window_order = {label: idx for idx, (label, _) in enumerate(TRAINING_WINDOWS)}
window_labels = {label: label.replace("_", " ") for label, _ in TRAINING_WINDOWS}

## In-Sample Overfitting-Capacity Training Loop

For each ticker and training-window length, the LSTM is trained on the selected training sequences and then predicts those same training sequences. There is no validation data, no test data, no early stopping, and no learning-rate callback.

In [ ]:
result_rows = []
prediction_frames = []
history_frames = []

for ticker in TICKERS:
    ticker_train_all = train_df.loc[train_df["Ticker"].eq(ticker)].sort_values("Date").copy()
    if ticker_train_all.empty:
        raise ValueError(f"No training data found for {ticker}.")

    for training_window, training_years in TRAINING_WINDOWS:
        selected_train = select_training_window(ticker_train_all, training_years)
        n_train_rows = len(selected_train)
        if n_train_rows < LOOKBACK + 20:
            print(f"Skipping {ticker} {training_window}: only {n_train_rows} rows.")
            continue

        feature_scaler = StandardScaler()
        scaled_feature_cols = [f"{col}_scaled" for col in FEATURE_COLS]
        selected_train = selected_train.copy()
        selected_train[scaled_feature_cols] = feature_scaler.fit_transform(selected_train[FEATURE_COLS])

        X_train, y_train, meta_train = make_training_sequences(
            selected_train,
            scaled_feature_cols,
            TARGET_COL,
            LOOKBACK,
        )
        if len(y_train) == 0:
            print(f"Skipping {ticker} {training_window}: zero training sequences.")
            continue

        target_scaler = StandardScaler()
        y_train_scaled = target_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

        tf.keras.utils.set_random_seed(SEED)
        model = build_lstm_model(input_shape=(LOOKBACK, len(FEATURE_COLS)), config=FIXED_CONFIG)

        print(
            f"Training {ticker} | {training_window} | rows={n_train_rows} | "
            f"sequences={len(y_train)} | epochs={EPOCHS}"
        )
        history = model.fit(
            X_train,
            y_train_scaled,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            shuffle=False,
            verbose=0,
        )

        y_pred_scaled = model.predict(X_train, verbose=0).reshape(-1)
        y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
        final_training_loss = float(history.history["loss"][-1])
        metrics = compute_train_metrics(y_train, y_pred)

        result_rows.append(
            {
                "ticker": ticker,
                "training_window": training_window,
                "training_years": training_years if training_years is not None else np.nan,
                "train_start": selected_train["Date"].min(),
                "train_end": selected_train["Date"].max(),
                "lookback": LOOKBACK,
                "n_features": len(FEATURE_COLS),
                "n_train_rows": n_train_rows,
                "n_train_sequences": len(y_train),
                "epochs_run": EPOCHS,
                "final_training_loss": final_training_loss,
                **metrics,
            }
        )

        pred_df = meta_train.copy()
        pred_df = pred_df.rename(columns={"Ticker": "ticker"})
        pred_df.insert(1, "training_window", training_window)
        pred_df.insert(2, "training_years", training_years if training_years is not None else np.nan)
        pred_df["y_true"] = y_train
        pred_df["y_pred"] = y_pred
        prediction_frames.append(pred_df)

        hist_df = pd.DataFrame(history.history)
        hist_df.insert(0, "epoch", np.arange(1, len(hist_df) + 1))
        hist_df.insert(0, "training_window", training_window)
        hist_df.insert(0, "ticker", ticker)
        history_frames.append(hist_df)

results_df = pd.DataFrame(result_rows).sort_values(["ticker", "training_window"]).reset_index(drop=True)
predictions_df = pd.concat(prediction_frames, ignore_index=True)
history_df = pd.concat(history_frames, ignore_index=True)

results_df["window_order"] = results_df["training_window"].map(window_order)
results_df = results_df.sort_values(["ticker", "window_order"]).drop(columns="window_order").reset_index(drop=True)
predictions_df["window_order"] = predictions_df["training_window"].map(window_order)
predictions_df = predictions_df.sort_values(["ticker", "window_order", "Input_End_Date"]).drop(columns="window_order").reset_index(drop=True)

display(results_df[[
    "ticker", "training_window", "n_train_rows", "n_train_sequences", "train_rmse", "train_r2",
    "train_variance_ratio", "train_mean_abs_prediction", "final_training_loss",
    "train_return_collapse_warning", "train_near_zero_prediction_warning",
]])

Training AAPL | 0.5_years | rows=128 | sequences=69 | epochs=200
Training AAPL | 1_year | rows=253 | sequences=194 | epochs=200
Training AAPL | 2_years | rows=505 | sequences=446 | epochs=200
Training AAPL | 3_years | rows=755 | sequences=696 | epochs=200
Training AAPL | 5_years | rows=1259 | sequences=1200 | epochs=200
Training AAPL | 8_years | rows=2015 | sequences=1956 | epochs=200
Training AAPL | 10_years | rows=2517 | sequences=2458 | epochs=200
Training AAPL | full_available_training_sample | rows=2708 | sequences=2649 | epochs=200
Training IBM | 0.5_years | rows=128 | sequences=69 | epochs=200
Training IBM | 1_year | rows=253 | sequences=194 | epochs=200


## Save Diagnostic Tables

In [ ]:
RESULTS_PATH = OUTPUT_DIR / "daily_lstm_overfit_capacity_results.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "daily_lstm_overfit_capacity_predictions.csv"
SUMMARY_PATH = OUTPUT_DIR / "daily_lstm_overfit_capacity_summary.csv"
HISTORY_PATH = OUTPUT_DIR / "daily_lstm_overfit_capacity_training_history.csv"

summary_rows = []
for ticker, group in results_df.assign(window_order=results_df["training_window"].map(window_order)).sort_values("window_order").groupby("ticker"):
    collapse = group.loc[group["train_return_collapse_warning"]]
    near_zero = group.loc[group["train_near_zero_prediction_warning"]]
    best_r2 = group.sort_values("train_r2", ascending=False).iloc[0]
    summary_rows.append(
        {
            "ticker": ticker,
            "best_train_r2_window": best_r2["training_window"],
            "best_train_r2": best_r2["train_r2"],
            "first_train_return_collapse_window": collapse.iloc[0]["training_window"] if not collapse.empty else None,
            "first_train_near_zero_prediction_window": near_zero.iloc[0]["training_window"] if not near_zero.empty else None,
            "smallest_window_train_variance_ratio": group.iloc[0]["train_variance_ratio"],
            "full_sample_train_variance_ratio": group.loc[group["training_window"].eq("full_available_training_sample"), "train_variance_ratio"].iloc[0]
            if group["training_window"].eq("full_available_training_sample").any()
            else np.nan,
            "smallest_window_train_mean_abs_prediction": group.iloc[0]["train_mean_abs_prediction"],
            "full_sample_train_mean_abs_prediction": group.loc[group["training_window"].eq("full_available_training_sample"), "train_mean_abs_prediction"].iloc[0]
            if group["training_window"].eq("full_available_training_sample").any()
            else np.nan,
        }
    )
summary_df = pd.DataFrame(summary_rows)

results_df.to_csv(RESULTS_PATH, index=False)
predictions_df.to_csv(PREDICTIONS_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)
history_df.to_csv(HISTORY_PATH, index=False)

print("Saved:")
for output_path in [RESULTS_PATH, PREDICTIONS_PATH, SUMMARY_PATH, HISTORY_PATH]:
    print(output_path)

display(summary_df)

## Saved In-Sample Diagnostic Plots

In [ ]:
plot_paths = []
plot_results_df = results_df.copy()
plot_results_df["window_order"] = plot_results_df["training_window"].map(window_order)
plot_results_df["training_window_label"] = plot_results_df["training_window"].map(window_labels)
plot_results_df = plot_results_df.sort_values(["ticker", "window_order"])

# 1. AAPL actual vs predicted training returns by training window.
aapl_pred_df = predictions_df.loc[predictions_df["ticker"].eq("AAPL")].copy()
aapl_pred_df["window_order"] = aapl_pred_df["training_window"].map(window_order)
aapl_pred_df = aapl_pred_df.sort_values(["window_order", "Input_End_Date"])
fig, axes = plt.subplots(4, 2, figsize=(16, 12), sharey=True)
axes = axes.ravel()
for ax, (training_window, _) in zip(axes, TRAINING_WINDOWS):
    window_df = aapl_pred_df.loc[aapl_pred_df["training_window"].eq(training_window)].sort_values("Input_End_Date")
    ax.plot(window_df["Input_End_Date"], window_df["y_true"], color="black", linewidth=0.9, alpha=0.65, label="Actual train return")
    ax.plot(window_df["Input_End_Date"], window_df["y_pred"], color="#1f77b4", linewidth=1.2, alpha=0.95, label="Predicted train return")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.7)
    ax.set_title(window_labels[training_window])
    ax.set_ylabel("Log return")
    if training_window == TRAINING_WINDOWS[0][0]:
        ax.legend(loc="upper right")
fig.suptitle("AAPL: actual vs predicted in-sample training returns by training window", fontsize=14)
plot_paths.append(save_plot("aapl_training_actual_vs_predicted_by_window.png"))

# 2-5. Metric line plots by training window and ticker.
metric_plots = [
    ("train_variance_ratio", "Training prediction std / actual std", "train_variance_ratio_by_training_window.png", 0.10),
    ("train_r2", "Training R-squared", "train_r2_by_training_window.png", None),
    ("train_mean_abs_prediction", "Training mean absolute prediction", "train_mean_abs_prediction_by_training_window.png", 0.0005),
    ("final_training_loss", "Final training loss", "final_training_loss_by_training_window.png", None),
]
for metric, ylabel, filename, reference in metric_plots:
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.lineplot(data=plot_results_df, x="window_order", y=metric, hue="ticker", marker="o", ax=ax)
    if reference is not None:
        ax.axhline(reference, color="red", linestyle="--", linewidth=1)
    ax.set_xticks(
        sorted(plot_results_df["window_order"].unique()),
        [window_labels[label] for label, _ in TRAINING_WINDOWS if label in set(plot_results_df["training_window"])],
        rotation=35,
        ha="right",
    )
    ax.set_xlabel("Training window")
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + " by training-window length")
    plot_paths.append(save_plot(filename))

# 6. Prediction distribution by training window.
dist_df = predictions_df.copy()
dist_df["training_window_label"] = dist_df["training_window"].map(window_labels)
g = sns.FacetGrid(dist_df, col="ticker", col_order=TICKERS, hue="training_window_label", height=4, aspect=1.25, sharex=False, sharey=False)
g.map_dataframe(sns.kdeplot, x="y_pred", fill=False, common_norm=False)
g.add_legend(title="Training window")
g.set_axis_labels("Predicted training return", "Density")
g.fig.suptitle("In-sample prediction distribution by training window", y=1.04)
plot_paths.append(save_plot("train_prediction_distribution_by_training_window.png"))

print("Saved plots:")
for path in plot_paths:
    print(path)

## Inline In-Sample Plots

These inline plots show only the in-sample overfitting diagnostic. They do not use validation or test predictions.

In [ ]:
inline_results_df = plot_results_df.copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
inline_specs = [
    ("train_variance_ratio", "Train variance ratio", 0.10),
    ("train_r2", "Train R-squared", None),
    ("train_mean_abs_prediction", "Train mean absolute prediction", 0.0005),
]
for ax, (metric, title, reference) in zip(axes, inline_specs):
    sns.lineplot(data=inline_results_df, x="window_order", y=metric, hue="ticker", marker="o", ax=ax)
    if reference is not None:
        ax.axhline(reference, color="red", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Training window")
    ax.set_ylabel(title)
    ax.set_xticks(
        sorted(inline_results_df["window_order"].unique()),
        [window_labels[label] for label, _ in TRAINING_WINDOWS if label in set(inline_results_df["training_window"])],
        rotation=35,
        ha="right",
    )
plt.tight_layout()
plt.show()

selected_inline_windows = ["1_year", "3_years", "5_years", "full_available_training_sample"]
aapl_inline_df = predictions_df.loc[
    predictions_df["ticker"].eq("AAPL")
    & predictions_df["training_window"].isin(selected_inline_windows)
].copy()
aapl_inline_df["window_order"] = aapl_inline_df["training_window"].map(window_order)
aapl_inline_df = aapl_inline_df.sort_values(["window_order", "Input_End_Date"])

fig, axes = plt.subplots(len(selected_inline_windows), 1, figsize=(14, 3.0 * len(selected_inline_windows)), sharex=False, sharey=True)
if len(selected_inline_windows) == 1:
    axes = [axes]
for ax, training_window in zip(axes, selected_inline_windows):
    window_df = aapl_inline_df.loc[aapl_inline_df["training_window"].eq(training_window)].sort_values("Input_End_Date")
    ax.plot(window_df["Input_End_Date"], window_df["y_true"], color="black", linewidth=0.9, alpha=0.65, label="Actual train return")
    ax.plot(window_df["Input_End_Date"], window_df["y_pred"], color="#1f77b4", linewidth=1.2, alpha=0.95, label="Predicted train return")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.7)
    ax.set_title(f"AAPL - {window_labels[training_window]}")
    ax.set_ylabel("Log return")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Input end date")
plt.tight_layout()
plt.show()

## Final Interpretation

This diagnostic is intentionally in-sample. It tests whether the LSTM architecture can still overfit the training sample as the number of training years increases.

If the LSTM can fit small windows but prediction variance shrinks for larger windows, this suggests that the daily return-regression target becomes harder to fit as the sample becomes more heterogeneous.

If prediction collapse happens even in-sample, then the issue is stronger than poor generalization: the model minimizes MSE by producing near-zero predictions even on the training data.

If the model can overfit all training windows but fails out-of-sample elsewhere, then the architecture has enough capacity, but the learned relationship does not generalize.

This directly answers the supervisor's question about how large the training sample can become before the LSTM returns basically always zero.